In [6]:
import random
from pathlib import Path

import cv2
import numpy as np

# --------------------------------------------------------
# Dataset Path
# --------------------------------------------------------

DATASET = Path("../dataset")

IMAGE_DIR = DATASET / "train" / "images"
LABEL_DIR = DATASET / "train" / "labels"

OUTPUT_DIR = Path("../outputs/samples")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------
# Classes
# --------------------------------------------------------

CLASS_NAMES = {
    0: "Calcium",
    1: "Healthy",
    2: "Magnesium",
    3: "Nitrogen",
    4: "Phosphorus",
    5: "Potassium"
}

# Different color for each class (BGR)
COLORS = {
    0: (0, 0, 255),        # Red
    1: (0, 255, 0),        # Green
    2: (255, 0, 255),      # Purple
    3: (255, 255, 0),      # Cyan
    4: (255, 0, 0),        # Blue
    5: (0, 165, 255)       # Orange
}

# --------------------------------------------------------
# Pick Random Images
# --------------------------------------------------------

images = list(IMAGE_DIR.glob("*.*"))

random.seed(42)

sample_images = random.sample(images, min(10, len(images)))

# --------------------------------------------------------
# Draw Labels
# --------------------------------------------------------

for image_path in sample_images:

    image = cv2.imread(str(image_path))

    h, w = image.shape[:2]

    label_path = LABEL_DIR / (image_path.stem + ".txt")

    if not label_path.exists():
        continue

    with open(label_path) as f:
        lines = f.readlines()

    for line in lines:

        values = line.strip().split()

        if len(values) < 7:
            continue

        class_id = int(values[0])

        coords = np.array(list(map(float, values[1:])))

        if len(coords) % 2 != 0:
            continue

        coords = coords.reshape(-1, 2)

        coords[:, 0] *= w
        coords[:, 1] *= h

        pts = coords.astype(np.int32)

        color = COLORS[class_id]

        # Draw polygon
        cv2.polylines(
            image,
            [pts],
            isClosed=True,
            color=color,
            thickness=2
        )

        # Transparent mask

        overlay = image.copy()

        cv2.fillPoly(
            overlay,
            [pts],
            color
        )

        image = cv2.addWeighted(
            overlay,
            0.30,
            image,
            0.70,
            0
        )

        x = pts[0][0]
        y = pts[0][1]

        cv2.putText(
            image,
            CLASS_NAMES[class_id],
            (x, y - 5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            color,
            2
        )

    save_path = OUTPUT_DIR / image_path.name

    cv2.imwrite(str(save_path), image)

    print(f"Saved {save_path}")

print("\nDone.")

Saved ..\outputs\samples\IMG20230605082516_jpg.rf.ce640524efa23e0d8050d9fc25b157bd.jpg
Saved ..\outputs\samples\0603201350_jpg.rf.ac1ab3d591d392750c9f16e004bd5512.jpg
Saved ..\outputs\samples\0603192346_jpg.rf.34b01f05ec1f21bab08e051961d3a04a.jpg
Saved ..\outputs\samples\IMG20230530065454_01_jpg.rf.ef3effab7f99794363f8fc88e0a7e4b6.jpg
Saved ..\outputs\samples\0605154250_jpg.rf.6548f25104634720f11729fcb329a8ad.jpg
Saved ..\outputs\samples\0605152127_jpg.rf.7416b36b8d08e9989227b32be02ea5d9.jpg
Saved ..\outputs\samples\0605141611_jpg.rf.e9f81434eea0bb0836311c19a7ae4f28.jpg
Saved ..\outputs\samples\0603200213_jpg.rf.38b3a3a8cd97636dfeaa527ccef1ace1.jpg
Saved ..\outputs\samples\IMG20230605082540_jpg.rf.eabe31d1bf50be8e0688eb9b29c12d73.jpg
Saved ..\outputs\samples\IMG20230602082234_jpg.rf.1f3095a3aabf9863cfbea8d5074c50b0.jpg

Done.
